# Strands Agent with AgentCore Memory Tutorial using Hooks

# Strands Agent 与 AgentCore Memory 教程（使用钩子）

## Overview

## 概述

This tutorial demonstrates how to build an intelligent personal assistant using Strands agents integrated with AgentCore Memory through hooks. The agent maintains conversation context and learns from interactions to provide personalized responses.

本教程演示如何使用通过钩子与 AgentCore Memory 集成的 Strands agents 构建智能个人助手。代理维护对话上下文并从交互中学习以提供个性化响应。

## Tutorial Details

## 教程详情

**Use Case**: Math Assistant

**用例**：数学助手

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long term Conversational                                                         |
| Agent type          | Math Assistant                                                                   |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | AgentCore Summary Strategy for Memory, Hooks for storing and retrieving Memory   |
| Example complexity  | Intermediate                                                                     |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 长期对话                                                                          |
| 代理类型             | 数学助手                                                                          |
| 代理框架             | Strands Agents                                                                   |
| LLM 模型             | Anthropic Claude Haiku 4.5                                                      |
| 教程组件             | AgentCore 记忆摘要策略、用于存储和检索记忆的钩子                                      |
| 示例复杂度           | 中级                                                                              |

You'll learn to:

您将学习：

- Set up AgentCore Memory with conversation summaries
- Create memory hooks for automatic storage and retrieval
- Build a Strands agent with persistent memory
- Test memory functionality across conversations

- 使用对话摘要设置 AgentCore Memory
- 创建用于自动存储和检索的记忆钩子
- 构建具有持久记忆的 Strands 代理
- 跨对话测试记忆功能

### Scenario Context

### 场景背景

In this example you'll create a Math Assistant example where you'd store summaries of the previous conversations. 

在本示例中，您将创建一个数学助手示例，其中您将存储以前对话的摘要。

Key features of this example:

本示例的关键功能：

- **Automatic Memory Storage**: Conversations are automatically saved
- **Context Retrieval**: Previous conversations inform current responses
- **Summary Generation**: Key information is extracted and summarized
- **Tool Integration**: Calculator tool for mathematical operations

- **自动记忆存储**：对话会自动保存
- **上下文检索**：以前的对话为当前响应提供信息
- **摘要生成**：提取并总结关键信息
- **工具集成**：用于数学运算的计算器工具

## Architecture

## 架构

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Prerequisites

## 前提条件

To execute this tutorial you will need:

要执行本教程，您将需要：

- Python 3.10+
- AWS credentials with Amazon Bedrock AgentCore Memory permissions
- Amazon Bedrock AgentCore SDK

- Python 3.10+
- 具有 Amazon Bedrock AgentCore Memory 权限的 AWS 凭证
- Amazon Bedrock AgentCore SDK

## Step 1: Environment set up

## 第一步：环境设置

Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

让我们开始导入所有必要的库并定义客户端以使本笔记本正常工作。

In [ ]:
!pip install -qr requirements.txt

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

In [ ]:
import os
import logging
from strands import Agent
from datetime import datetime
from strands_tools import calculator
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("memory-tutorial")

# Configuration - replace with your values
REGION = os.getenv('AWS_REGION', 'us-west-2')
ROLE_ARN = "<<INSERT-YOUR-IAM-ROLE>>"
ACTOR_ID = f"actor-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"tutorial-{datetime.now().strftime('%Y%m%d%H%M%S')}"

## Step 2: Create Memory Resource

## 第二步：创建记忆资源

In this step, we're creating our memory resource with a summary strategy. This resource will store and organize our conversation data. The strategy we're defining will automatically generate summaries of conversations and store them in organized namespaces.

在此步骤中，我们将使用摘要策略创建记忆资源。此资源将存储和组织我们的对话数据。我们定义的策略将自动生成对话摘要并将其存储在有组织的命名空间中。

Firstly, lets create a custom prompt for the math assistant.

首先，让我们为数学助手创建一个自定义提示。

In [ ]:
CUSTOM_PROMPT = """
Your task is to extract math learning data from the user's conversations. You store the progress of the user in a memory system to understand their math level and help them progress.

You are tasked with analyzing conversations to extract the user's math learning patterns. You'll be analyzing two sets of data: 

<past_conversation> 
[Past conversations between the user and math tutor will be placed here for context] 
</past_conversation> 

<current_conversation> 
[The current conversation between the user and math tutor will be placed here] 
</current_conversation> 

Your job is to identify and categorize the user's math learning profile:
- Extract the user's current math level from problems they solve correctly/incorrectly
- Extract the user's preferred learning style from how they ask questions and respond to explanations
- Extract topic strengths and weaknesses from their performance patterns
- Track learning progress and identify areas needing reinforcement
"""

In [ ]:
from botocore.exceptions import ClientError

# Initialize Memory Client
client = MemoryClient(region_name=REGION)
memory_name = "MathAssistant"
# Define memory strategy for conversation summaries
strategies = [
    {
        StrategyType.CUSTOM.value: {
            "name": "CustomSemanticMemory",
            "description": "Captures facts from conversations",
            "namespaces": ["/students/math/{actorId}"],
            "configuration" : {
                "semanticOverride" : {
                    "extraction" : {
                        "modelId" : "global.anthropic.claude-haiku-4-5-20251001-v1:0",
                        "appendToPrompt": CUSTOM_PROMPT
                    }
                },
    }}}
]

# Create memory resource
try:
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=strategies, # Use the defined long term strategies
        description="Memory for tutorial agent",
        event_expiry_days=30,
        memory_execution_role_arn=ROLE_ARN,
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

## Step 3: Create Memory Hook Provider

## 第三步：创建记忆钩子提供程序

This step defines our custom `MemoryHookProvider` class that automates memory operations. Hooks are special functions that run at specific points in an agent's execution lifecycle. The memory hook we're creating serves two primary functions:

此步骤定义了我们的自定义 `MemoryHookProvider` 类，它自动化记忆操作。钩子是在代理执行生命周期的特定点运行的特殊函数。我们创建的记忆钩子提供两个主要功能：

1. **Retrieve Memories**: Automatically fetches relevant past conversations when a user sends a message
2. **Save Memories**: Stores new conversations after the agent responds

1. **检索记忆**：当用户发送消息时自动获取相关的过去对话
2. **保存记忆**：在代理响应后存储新对话

This creates a seamless memory experience without manual management.

这创建了一种无需手动管理的无缝记忆体验。

In [ ]:
class MemoryHookProvider(HookProvider):
    """Hook provider for automatic memory management"""
    
    def __init__(self, memory_id: str, client: MemoryClient):
        self.memory_id = memory_id
        self.client = client
    
    def retrieve_memories(self, event: MessageAddedEvent):
        """Retrieve relevant memories before processing user message"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_message = messages[-1]["content"][0].get("text", "")
            
            try:
                # Get actor_id from agent state
                actor_id = event.agent.state.get("actor_id")
                if not actor_id:
                    logger.warning("Missing actor_id in agent state")
                    return
                
                namespace = f"/students/math/{actor_id}"
                
                # Retrieve relevant memories
                memories = self.client.retrieve_memories(
                    memory_id=self.memory_id,
                    namespace=namespace,
                    query=user_message
                )
                
                # Extract memory content
                memory_context = []
                for memory in memories:
                    if isinstance(memory, dict):
                        content = memory.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            if text:
                                memory_context.append(text)
                
                # Inject memories into user message
                if memory_context:
                    context_text = "\n".join(memory_context)
                    original_text = messages[-1]["content"][0].get("text", "")
                    messages[-1]["content"][0]["text"] = (
                        f"{original_text}\n\nPrevious context: {context_text}"
                    )
                    logger.info(f"Retrieved {len(memory_context)} memories")
                    
            except Exception as e:
                logger.error(f"Failed to retrieve memories: {e}")
    
    def save_memories(self, event: AfterInvocationEvent):
        """Save conversation after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last user and assistant messages
                user_msg = None
                assistant_msg = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break
                
                if user_msg and assistant_msg:
                    # Get session info from agent state
                    actor_id = event.agent.state.get("actor_id")
                    session_id = event.agent.state.get("session_id")
                    
                    if not actor_id or not session_id:
                        logger.warning("Missing actor_id or session_id in agent state")
                        return
                    
                    # Save conversation
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=actor_id,
                        session_id=session_id,
                        messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")]
                    )
                    logger.info("Saved conversation to memory")
                    
        except Exception as e:
            logger.error(f"Failed to save memories: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_memories)
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("Memory hooks registered")

## Step 4: Create Agent with Memory

## 第四步：创建带有记忆的代理

Now we're creating our Strands agent and connecting it with our memory hook provider. This agent will have two key capabilities:

现在我们正在创建 Strands 代理并将其与我们的记忆钩子提供程序连接。此代理将具有两个关键功能：

1. **Memory Integration**: The memory hooks we created will enable automatic context retrieval
2. **Calculator Tool**: The agent can perform mathematical operations when needed

1. **记忆集成**：我们创建的记忆钩子将启用自动上下文检索
2. **计算器工具**：代理可以在需要时执行数学运算

This combination creates a personal assistant that both remembers past interactions and can perform useful calculations.

这种组合创建了一个既能记住过去交互又能执行有用计算的个人助手。

In [ ]:
# Create memory hook provider
memory_hooks = MemoryHookProvider(memory_id, client)

# Create agent with memory hooks and calculator tool
agent = Agent(
    hooks=[memory_hooks],
    model = "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[calculator],
    system_prompt="You are a helpful personal math tutor. You assist users in solving math problems and provide personalized assistance.",
    state={"actor_id": ACTOR_ID, "session_id": SESSION_ID}
)

print("✅ Agent created with memory hooks.")

**We have our agent set up ! Let's test it now.**

**我们的代理已设置好！现在让我们测试它。**

## Test Memory Functionality

## 测试记忆功能

In this section, we'll test the agent's memory capabilities through a series of interactions. We'll observe how the agent builds context over time and recalls previous interactions.

在本节中，我们将通过一系列交互测试代理的记忆能力。我们将观察代理如何随时间构建上下文并回忆以前的交互。

First, let's introduce ourselves to the agent and ask a math question:

首先，让我们向代理介绍自己并问一个数学问题：

In [ ]:
# First interaction - introduce yourself
response1 = agent("Hi, I'm John and I just enrolled in Discrete Math course. Help me solve this: How many ways can I arrange 5 books on a shelf?")
print(f"Agent: {response1}")

Let's give the agent another calculation task:

让我们给代理另一个计算任务：

In [ ]:
# Second interaction - another calculation
response2 = agent("I learn better with step-by-step explanation with example questions. Can you explain modular arithmetic? What's 17 mod 5?")
print(f"Agent: {response2}")

Now, let's see if the agent remembers who we are.

现在，让我们看看代理是否记得我们是谁。

**Note:** Give a ~20 sec pause here to allow some time for the memory to be extracted, consolidated and stored.

**注意：**在此处暂停约 20 秒，以便有时间提取、整合和存储记忆。

In [ ]:
# Third interaction - test memory recall
response3 = agent("I got that right! What's the immediate next step that I should study after modular arithmetic?")
print(f"Agent: {response3}")

Finally, let's check if the agent remembers our calculation history:

最后，让我们检查代理是否记得我们的计算历史：

In [ ]:
# Fourth interaction - test context awareness
response4 = agent("This is too hard, can we try something easier?")
print(f"Agent: {response4}")

### Verify Memory Storage

### 验证记忆存储

As a final step, we'll verify that our conversations have been properly stored in AgentCore Memory. This demonstrates that the memory hooks are working correctly and the agent can access this information in future interactions.

作为最后一步，我们将验证我们的对话是否已正确存储在 AgentCore Memory 中。这表明记忆钩子正在正确工作，代理可以在未来的交互中访问此信息。

In [ ]:
# Check stored memories
try:
    memories = client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"/students/math/{ACTOR_ID}",
        query="mathematics calculations"
    )
    
    print(f"\n📚 Found {len(memories)} memories:")
    for i, memory in enumerate(memories, 1):
        if isinstance(memory, dict):
            content = memory.get('content', {})
            if isinstance(content, dict):
                text = content.get('text', '')[:200] + "..."
                print(f"{i}. {text}")
                
except Exception as e:
    print(f"Error retrieving memories: {e}")

Tutorial completed! 🎉

教程完成！🎉

Key takeaways:

关键收获：

- Memory hooks automatically store and retrieve conversation context
- Agents can maintain state across multiple interactions
- AgentCore Memory provides semantic search for relevant context
- Tools can be combined with memory for enhanced functionality

- 记忆钩子自动存储和检索对话上下文
- 代理可以在多个交互中维护状态
- AgentCore Memory 提供相关上下文的语义搜索
- 工具可以与记忆结合以增强功能

## Clean Up

## 清理

### Optional: Delete Memory Resource

### 可选：删除记忆资源

After completing the tutorial, you may want to delete the memory resource to avoid incurring unnecessary costs. The following code is provided for cleanup but is commented out by default.

完成教程后，您可能希望删除记忆资源以避免产生不必要的费用。以下代码用于清理，但默认情况下已注释掉。

In [ ]:
# Uncomment to delete the memory resource
# try:
#     client.delete_memory_and_wait(memory_id=memory_id)
#     print(f"✅ Deleted memory resource: {memory_id}")
# except Exception as e:
#     print(f"Error deleting memory: {e}")